In [5]:
# [CELL 1] STUDENT CONFIGURATION
from pathlib import Path

LAST_NAME = 'Magalong'
STUDENT_ID = 'TUPM-26-1134'

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGTH = len(LAST_NAME)

print('Seed:', SEED_DIGIT)
print('ID Sum:', ID_SUM)
print('Name Length:', NAME_LENGTH)

Seed: 4
ID Sum: 17
Name Length: 8


In [6]:
# [CELL 2] CREATE ACTIVITY WORKSPACE
WORK_DIR = Path('.')
DATA_DIR = WORK_DIR / 'data'
OUTPUT_DIR = WORK_DIR / 'output'
BACKUP_DIR = WORK_DIR / 'backup'

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

print('Student:', LAST_NAME)
print('Student ID:', STUDENT_ID)
print('Workspace:', WORK_DIR.resolve())
print('Data folder exists:', DATA_DIR.exists())
print('Output folder exists:', OUTPUT_DIR.exists())
print('Backup folder exists:', BACKUP_DIR.exists())

Student: Magalong
Student ID: TUPM-26-1134
Workspace: C:\Users\Paul\Documents\TUPM-BSECE\1C\AECEL1\Lab\Lab6_File_Handling_Magalong\Lab6_Additional_Exercises
Data folder exists: True
Output folder exists: True
Backup folder exists: True


In [7]:
# [CELL 3] EXERCISE 3 PARAMETERS
import csv
import json
import shutil
from collections import Counter

dataset_path = DATA_DIR / 'Global_Cybersecurity_Threats_2015-2024.csv'

target_year = 2015 + SEED_DIGIT
loss_field = 'Financial Loss (in Million $)'
count_field = 'Attack Type'

print('Dataset:', dataset_path)
print('Dataset exists:', dataset_path.exists())
print('Target year:', target_year)

Dataset: data\Global_Cybersecurity_Threats_2015-2024.csv
Dataset exists: True
Target year: 2019


In [8]:
# [CELL 4] READ DATASET AND SELECT INCIDENTS
total_records = 0
invalid_loss_records = 0
selected_incidents = []

with open(dataset_path, 'r', encoding='utf-8-sig', newline='') as file:
    reader = csv.DictReader(file)
    fieldnames = reader.fieldnames

    for row in reader:
        total_records += 1

        try:
            year = int(row['Year'])
        except (KeyError, ValueError):
            continue

        if year != target_year:
            continue

        try:
            loss = float(row[loss_field])
        except (KeyError, ValueError):
            invalid_loss_records += 1
            continue

        selected_incidents.append((row, loss))

print('Total records in dataset:', total_records)
print('Incidents in target year:', len(selected_incidents))
print('Invalid financial values skipped:', invalid_loss_records)

Total records in dataset: 3000
Incidents in target year: 263
Invalid financial values skipped: 0


In [9]:
# [CELL 5] COUNT ATTACK TYPES AND COMPUTE LOSSES
attack_counts = Counter(row[count_field].strip() for row, _ in selected_incidents)

total_loss = sum(loss for _, loss in selected_incidents)
average_loss = total_loss / len(selected_incidents) if selected_incidents else 0

print('Count per attack type:')
for attack, count in attack_counts.most_common():
    print(f'  {attack}: {count}')

print('Total financial loss (USD millions):', round(total_loss, 2))
print('Average financial loss (USD millions):', round(average_loss, 2))

Count per attack type:
  DDoS: 48
  Phishing: 45
  SQL Injection: 45
  Man-in-the-Middle: 44
  Malware: 43
  Ransomware: 38
Total financial loss (USD millions): 13134.69
Average financial loss (USD millions): 49.94


In [10]:
# [CELL 6] FIND FIVE COSTLIEST INCIDENTS
# sorted() is stable, so ties keep the original dataset order
costliest = sorted(selected_incidents, key=lambda item: -item[1])[:5]

for rank, (row, loss) in enumerate(costliest, 1):
    print(f"{rank}. {row[count_field]} | {row.get('Country', 'N/A')} | {row.get('Target Industry', 'N/A')} | {loss} M")

1. Phishing | Germany | Healthcare | 99.97 M
2. Man-in-the-Middle | Russia | Retail | 99.53 M
3. Phishing | UK | Education | 99.45 M
4. Ransomware | Germany | Government | 99.3 M
5. SQL Injection | Australia | Government | 99.2 M


In [11]:
# [CELL 7] EXPORT FILTERED CSV AND TOP FIVE JSON
filtered_csv_path = OUTPUT_DIR / f'{LAST_NAME}_{STUDENT_ID}_incidents_{target_year}.csv'
top_five_path = OUTPUT_DIR / f'{LAST_NAME}_{STUDENT_ID}_top5_incidents.json'

with open(filtered_csv_path, 'w', encoding='utf-8', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(row for row, _ in selected_incidents)

top_five_records = [{'rank': rank, **row} for rank, (row, _) in enumerate(costliest, 1)]

with open(top_five_path, 'w', encoding='utf-8') as file:
    json.dump(top_five_records, file, indent=4)

print('Created:', filtered_csv_path)
print('Created:', top_five_path)

Created: output\Magalong_TUPM-26-1134_incidents_2019.csv
Created: output\Magalong_TUPM-26-1134_top5_incidents.json


In [12]:
# [CELL 8] WRITE JSON SUMMARY
summary_path = OUTPUT_DIR / f'{LAST_NAME}_{STUDENT_ID}_incident_summary.json'

summary = {
    'last_name': LAST_NAME,
    'student_id': STUDENT_ID,
    'target_year': target_year,
    'total_records': total_records,
    'incidents_in_target_year': len(selected_incidents),
    'invalid_financial_values': invalid_loss_records,
    'count_per_attack_type': dict(attack_counts),
    'total_financial_loss_usd_millions': round(total_loss, 2),
    'average_financial_loss_usd_millions': round(average_loss, 2)
}

with open(summary_path, 'w', encoding='utf-8') as file:
    json.dump(summary, file, indent=4)

print('Created:', summary_path)

Created: output\Magalong_TUPM-26-1134_incident_summary.json


In [13]:
# [CELL 9] BACKUP JSON FILE AND VERIFY
backup_path = BACKUP_DIR / top_five_path.name

shutil.copy2(top_five_path, backup_path)

backup_exists = backup_path.exists()
sizes_match = backup_exists and backup_path.stat().st_size == top_five_path.stat().st_size

print('Original:', top_five_path)
print('Backup:', backup_path)
print('Backup exists:', backup_exists)
print('Backup size matches original:', sizes_match)
print('Backup verification:', 'VERIFIED' if sizes_match else 'CHECK')

Original: output\Magalong_TUPM-26-1134_top5_incidents.json
Backup: backup\Magalong_TUPM-26-1134_top5_incidents.json
Backup exists: True
Backup size matches original: True
Backup verification: VERIFIED


In [14]:
# [CELL 10] GENERATE TEXT REPORT
report_path = OUTPUT_DIR / f'{LAST_NAME}_{STUDENT_ID}_incident_report.txt'

with open(report_path, 'w', encoding='utf-8') as report:
    report.write('CYBERSECURITY INCIDENT REPORT\n')
    report.write('-----------------------------\n')
    report.write(f'Last Name: {LAST_NAME}\n')
    report.write(f'Student ID: {STUDENT_ID}\n')
    report.write(f'Target Year: {target_year}\n')
    report.write(f'Incidents in Target Year: {len(selected_incidents)}\n\n')

    report.write('Incident Count per Attack Type:\n')
    if attack_counts:
        for attack, count in attack_counts.most_common():
            report.write(f'  {attack}: {count}\n')
    else:
        report.write('  No incidents found\n')

    report.write('\nFinancial Losses (USD millions):\n')
    report.write(f'  Total: {total_loss:.2f}\n')
    report.write(f'  Average: {average_loss:.2f}\n\n')

    report.write('Five Costliest Incidents:\n')
    for rank, (row, loss) in enumerate(costliest, 1):
        report.write(f"  {rank}. {row[count_field]} | {row.get('Country', 'N/A')} | {row.get('Target Industry', 'N/A')} | {loss} M\n")

    report.write('\nOutput Files:\n')
    report.write(f'  Filtered CSV: {filtered_csv_path.name}\n')
    report.write(f'  JSON Summary: {summary_path.name}\n')
    report.write(f'  Top Five JSON: {top_five_path.name}\n')
    report.write(f'  Backup File: {backup_path.name}\n')

print('Report created:', report_path)

Report created: output\Magalong_TUPM-26-1134_incident_report.txt


In [15]:
# [CELL 11] FINAL OUTPUT
print('ASSESSMENT DATA - CYBERSECURITY INCIDENT REPORT')
print('-----------------------------------------------')
print('Target year:', target_year)
print('Total records in dataset:', total_records)
print('Incidents in target year:', len(selected_incidents))
print('Count per attack type:', dict(attack_counts))
print('Total financial loss (USD millions):', round(total_loss, 2))
print('Average financial loss (USD millions):', round(average_loss, 2))
print('Five costliest incidents:')
for rank, (row, loss) in enumerate(costliest, 1):
    print(f'  {rank}. {row[count_field]} - {loss} M')
print('Filtered CSV file:', filtered_csv_path.name)
print('JSON summary file:', summary_path.name)
print('Backup verification:', 'VERIFIED' if sizes_match else 'CHECK')
print('Report file:', report_path.name)

ASSESSMENT DATA - CYBERSECURITY INCIDENT REPORT
-----------------------------------------------
Target year: 2019
Total records in dataset: 3000
Incidents in target year: 263
Count per attack type: {'Phishing': 45, 'Ransomware': 38, 'Malware': 43, 'Man-in-the-Middle': 44, 'SQL Injection': 45, 'DDoS': 48}
Total financial loss (USD millions): 13134.69
Average financial loss (USD millions): 49.94
Five costliest incidents:
  1. Phishing - 99.97 M
  2. Man-in-the-Middle - 99.53 M
  3. Phishing - 99.45 M
  4. Ransomware - 99.3 M
  5. SQL Injection - 99.2 M
Filtered CSV file: Magalong_TUPM-26-1134_incidents_2019.csv
JSON summary file: Magalong_TUPM-26-1134_incident_summary.json
Backup verification: VERIFIED
Report file: Magalong_TUPM-26-1134_incident_report.txt
